In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [2]:
import os

In [3]:
load_dotenv(override=True)

True

### Reading LinkedIN profile and summary to create a digital twin

In [4]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [5]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

In [6]:
llm2 = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPEN_ROUTER_API_KEY"),
)

In [18]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Aryan"},
    {"role": "assistant", "content": "Well hi there, Aryan. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = llm2.chat.completions.create(model="nvidia/nemotron-nano-9b-v2:free", messages=messages)
print(response.choices[0].message.content)

## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [19]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

### Chat function which LLM automatically calls using Gradio Lib
----

In [20]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = llm2.chat.completions.create(model="nvidia/nemotron-nano-9b-v2:free", messages=messages)
    return response.choices[0].message.content

In [ ]:
display(Markdown(chat("Please summarize who you are and do you like Pizza ?", [])))

### Now it's Gradio lib's turn
---
Gradio is an open-source Python library that allows you to quickly create user interfaces for machine learning models, APIs, or any arbitrary Python function.

In [ ]:
# gr.ChatInterface(chat).launch(inbrowser=True)

# And now - TOOLS!

Let's start with a function...

In [21]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

## Step 1 - write some json to describe the tool


In [22]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
    
}

In [23]:
tools = [{"type": "function", "function": record_email_tool_json}]
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

## Step 2 - a new chat() function

This is where we implement the tool call.

In [39]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = llm2.chat.completions.create(model="nvidia/nemotron-nano-9b-v2:free", messages=messages, tools=tools)
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = llm2.chat.completions.create(model="nvidia/nemotron-nano-9b-v2:free", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [30]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called to record an email: a@nim.com


## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [42]:
messages

[{'role': 'system', 'content': 'You are a snarky, witty assistant'},
 {'role': 'user', 'content': 'Hi, my name is Aryan'},
 {'role': 'assistant',
  'content': "Well hi there, Aryan. It's nice to meet you."},
 {'role': 'user', 'content': "What's my name?"}]

In [41]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = llm2.chat.completions.create(model="nvidia/nemotron-nano-9b-v2:free", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = llm2.chat.completions.create(model="nvidia/nemotron-nano-9b-v2:free", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [43]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called to record an email: 123@nim.com
Tool called to record an email: 124@nim.com
Tool called to record an email: 125@nim.com
